# Khảo sát dữ liệu lá cà phê

Notebook trả lời tám câu hỏi về dữ liệu, ảnh và đặc trưng. Mọi ảnh và phép tính từ đặc trưng chỉ dùng ID của tập train. Test trong bảng chia là số suy ra từ tổng ảnh dùng được trừ train.

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "configs/default.yaml").is_file():
    parent = project_root.parent
    if parent == project_root:
        raise FileNotFoundError("Cannot locate configs/default.yaml")
    project_root = parent

os.chdir(project_root)
sys.path[:0] = [str(project_root / "src"), str(project_root)]

import matplotlib.pyplot as plt
import pandas as pd
from threadpoolctl import threadpool_limits

from coffee_leaf.analysis import plots, summaries
from coffee_leaf.config import load_config
from scripts import run_eda


def display_figure(figure):
    display(figure)
    plt.close(figure)


configuration = load_config(run_eda.CONFIG_PATH)
label_table, image_count = run_eda.load_label_table(configuration)
training = run_eda.load_train_features(run_eda.TRAIN_FEATURE_PATH, configuration)
train_paths = run_eda.build_train_image_paths(label_table, training, configuration)
label_by_code = configuration["label_by_code"]
features = training["features"]
train_labels = training["labels"]
feature_metadata = [training[key] for key in ("feature_names", "feature_groups", "feature_regions")]

## 1. Dữ liệu và nhãn có bao nhiêu mẫu?

CSV có 1.747 dòng, 1.401 dòng khớp tên tệp và 1.342 ảnh dùng được. Tất cả 1.006 ảnh train có kích thước gốc 2.048 × 1.024 px. Một dòng chỉ được dùng khi mã chính nằm trong 0–4 và có ảnh tương ứng.

In [ ]:
overview = summaries.summarize_dataset(label_table, label_by_code)
display(overview)
print("JPG files:", image_count)
image_sizes = run_eda.read_train_image_sizes(train_paths)
display(summaries.summarize_image_sizes(image_sizes))

## 2. Nhãn mất cân bằng đến mức nào?

Lớp lớn nhất là `rust` với 465 ảnh, so với 136 ảnh `cercospora`, gấp 3,42 lần. Bảng có 1.006 train và 336 test suy ra. Lớp ít ảnh có ít ví dụ để mô hình học và để ước lượng điểm.

In [ ]:
split_counts = summaries.summarize_split(overview, train_labels)
display(split_counts)
display_figure(plots.plot_class_distribution(split_counts))

## 3. Nhãn chính có che dấu các cờ bệnh khác không?

Có 372 dòng bật từ hai cờ bệnh, gồm 315 dòng trong các mã 0–4. Có 52 trong 147 dòng mã `cercospora` bật thêm cờ `rust`. Một lá có thể mang nhiều dấu hiệu dù pipeline lưu một nhãn chính.

In [ ]:
crosstab, multilabel_counts = summaries.summarize_multilabel(label_table, label_by_code)
display(crosstab)
display(pd.DataFrame([multilabel_counts]))
display_figure(plots.plot_multilabel(crosstab))

## 4. Severity thực tế nằm trong khoảng nào?

Trong 1.685 dòng CSV có nhãn, `healthy` chỉ có giá trị 0 và bốn lớp bệnh có trung vị 1. `leaf_miner` có cực đại 3, ba lớp bệnh còn lại có cực đại 4. Đây là giá trị ghi trong CSV và là nguồn tham chiếu yếu về độ nặng.

In [ ]:
severity = summaries.summarize_severity(label_table, label_by_code)
severity_samples = summaries.build_severity_samples(label_table, label_by_code)
display(severity)
display_figure(plots.plot_severity_by_class(severity_samples))

## 5. Mặt nạ khoanh được phần nào của ảnh?

Xem 15 ảnh train, 3 ảnh mỗi lớp, và 10 cặp ảnh gốc–ảnh phủ mặt nạ, 2 cặp mỗi lớp. Xanh lá đánh dấu vùng lá, cam đánh dấu vùng khác màu bị nghi tổn thương. Các mẫu giúp kiểm tra trực quan và chưa thay cho đo độ chính xác bằng mặt nạ gán tay.

In [ ]:
sample_images = run_eda.load_sample_images(training, train_paths, configuration)
with threadpool_limits(limits=1):
    segmented_images = run_eda.segment_sample_images(sample_images)

selected_rows = []
for label, images in sample_images.items():
    for position, (image_id, image) in enumerate(images):
        has_overlay = position < run_eda.SEGMENTED_IMAGES_PER_CLASS
        selected_rows.append({"label": label, "image_id": image_id, "overlay": has_overlay})
selection_table = pd.DataFrame(selected_rows)
display(selection_table)
display_figure(plots.plot_sample_images(sample_images))
display_figure(plots.plot_segmentation_overlays(segmented_images))

## 6. Vùng nghi tổn thương chiếm bao nhiêu diện tích lá?

Trung vị trên train là 4,22% ở `healthy` và 9,77% ở `leaf_miner`. Có 37 ảnh train có cả khối 135 giá trị vùng nghi tổn thương bằng 0. Số cũ 44 chưa rõ phạm vi. Vùng khác màu trên lá khỏe vẫn có thể bị khoanh.

In [ ]:
ratios = summaries.extract_lesion_ratios(features, training["feature_names"])
ratio_table = summaries.summarize_lesion_ratios(ratios, train_labels)
zero_blocks = summaries.summarize_zero_lesion_blocks(features, training["feature_regions"], train_labels)
ratio_samples = pd.DataFrame({"label": train_labels, "lesion_ratio": ratios})
display(ratio_table)
display(zero_blocks)
display_figure(plots.plot_lesion_ratio_by_class(ratio_samples))

## 7. Bao nhiêu cột hằng và bao nhiêu cặp tương quan mạnh?

Có 62 cột hằng, tất cả bằng 0. Trên 341 cột còn lại có 237 cặp Pearson |r| ≥ 0,9. Có 146 cột, tương đương 42,82%, thuộc ít nhất một cặp như vậy. Sự phụ thuộc này là một lý do thực nghiệm để không ưu tiên Naive Bayes, dù tương quan gộp không trực tiếp đo độc lập có điều kiện theo lớp.

In [ ]:
constant_columns = summaries.summarize_constant_columns(features, *feature_metadata)
correlation, ordered_metadata = summaries.build_correlation_matrix(features, *feature_metadata)
correlation_summary, top_pairs = summaries.summarize_correlation_pairs(correlation)
display(constant_columns)
display(correlation_summary)
display(top_pairs)
display_figure(plots.plot_feature_correlation(correlation, ordered_metadata))

## 8. Hai thành phần PCA có tách các lớp ra không?

PC1 giải thích 19,72%, PC2 giải thích 11,56%, tổng 31,28%. Cần 63 thành phần để đạt 90%. Trên hình hai chiều, `cercospora` chồng lấn với các lớp khác, trong đó có `rust` và `leaf_miner`. Hình giữ một phần biến thiên, nên chồng lấn trên hình chưa đủ để kết luận về phân loại trên 403 cột.

In [ ]:
with threadpool_limits(limits=1):
    scores, variance, component_count = summaries.compute_pca(features, configuration["seed"])

display(variance.head(2))
print("Components for 90% variance:", component_count)
display_figure(plots.plot_pca_scatter(scores, train_labels, variance))